# E009 V15 — Tiếp tục campaign đã dừng

**Recovery đã được owner duyệt ngày 07/11/2026:** cap research **45 phút/run**, tổng **12 giờ** và pool research **500 phút** giữ nguyên. Cài đặt model, dữ liệu, seeds và queue không đổi. Sáu smoke phải được xác minh checkpoint/full-resume/reload/runtime; không train lại smoke. Cho phép hoàn tất xác minh sau cửa sổ smoke gốc.

1. Giữ nguyên `e009-colab-execution-bundle.zip` trên **My Drive/E009/**. **Không phải upload lại ZIP 1,27 GB.** Notebook mang extension recovery và authorization có SHA riêng; source đã tạo checkpoints giữ nguyên.
2. Xác nhận runtime/process cũ đã dừng, rồi đổi **`OLD_RUNTIME_STOPPED = True`**. `RESUME = True` đã được đặt sẵn. Chọn **A100-SXM4-80GB**, cùng runtime/environment; launcher dừng nếu khác fingerprint. Không chạy song song hai phiên.
3. Bấm **Run all**, mount đúng Drive. Kernel 3.13 được hỗ trợ, launcher/train vẫn dùng CPython **3.12.12** thật. Cần **50 GB local disk trống**; environment closure đầu tiên được dùng để restore.
4. Đồng hồ giữ từ **01:33:15 ngày 07/11/2026 (Việt Nam)**: research không quá **11:43:15**, tổng không quá **13:33:15**. Bao gồm gián đoạn/recovery, không reset. Số run hoàn tất phụ thuộc budget còn lại; không bảo đảm đủ 25 run. Hết budget thì dừng, không chạy như campaign mới.
5. Output vẫn ở **My Drive/E009/E009-V15-COLAB-20261106/**; input extension/authorization ở bootstrap hiện có. Giữ journal, mirror manifest, checkpoints và recovery. Cell in ZIP metadata khi xong/lỗi; tải về để audit. Notebook tự `runtime.unassign()` khi kết thúc.

**Chỉ dành cho campaign đã dừng vì projection B448 vượt 20 phút.** Receipt PASS chỉ được tạo sau khi kiểm tra thực tế các checkpoint cũ; notebook không tự ghi PASS từ metadata. Test/promotion và thay model prototype vẫn đóng.


In [ ]:
import hashlib, json, math, os, shutil, signal, subprocess, sys, threading, time
from pathlib import Path
from google.colab import drive, runtime

RESUME = True
OLD_RUNTIME_STOPPED = False
EXPECTED_SHA256 = '8125a39ba893624730193a219a4b3ab340592eb66e2f687f3bc1b55b3f7b95b1'
PYTHON_VERSION = '3.12.12'
UV_VERSION = '0.9.7'
RECOVERY_PAYLOAD = 'eNq1O9ty28ixv4IoL6RDUiBI8ObSVimONnGd3bXL8m5VSlKhBsBAxAoEEAwomXHp+TyfbzhfkOc85vxI/uR0zw2DGynbG2/V2sT09PT09H16Pp/RTyVNWZylk/xwtrHObvl/7+M0paF1Zdtrq6BB9kiLg6VBX1vlllrBlgYPeRan5TgvsnAfxOm9lZPggdxTi5XkwKx4t9uXxE/o5Da9Td+lyYHPJDlMeAT8OS3GxT61yngH+EgOOEl6T9nEelfE93FKEqssSJzS4jzI0ii+tx5JEoekzAo2uk1hKp8ZhxT+UR5GVpDt8oSWNORot4RtKbNIGvJlC8ooKYKt9bc93ePPHaC2yL7cZkVckjJ+RDo/AmSaldTPsgdcPWV5VpQMMMTMiuIE9kZzUpCSwm6iIttx3JmiNy/iRxiz/H0ain0LlsL+ObDnRftyX1DPA+4gZiAPloPVs5QhlPpa3MMqjOoPO1Ju9Q/ctkSYw/ck9hW29xxM0HXIKVPfv9+nAa7xET5W43hkEuAyPRjLA4eDrSaaxB79RHbeLktjGIFZE+A0/JrQooCzUDj+REryJkuBa0F5hSMn53NBSEkaUIWDbYnjLjzkdP9sLhUG8QXN8XRTwUcPfgDzQNCYJf9ZnMY1IUUZR0C53k6QhdSrhOsJpIR6v7IsfQGygOxyEt+nnhZSgfNtGoDgIaXJjzGyaGTlIKfUCykJExD1F+Du2a1cAQ71RTiEMngEpPbAYlaxkmXJI/VIsfsSNGq/Cs0HOfBGfh8BYhKOQKMo9ejf9iRB6bpNQxrByGPMaOj5+/CelgPx18YK46C8YSVwCGTzbmiNv2t82tymFvwRGoZ6Wz5lFhgIYCezdnsGWnQPy722KLdfGShqURmfJN7FpQUbB83gZiecKF1FrAUFNU2tz+IX/nn1SlA2qj7dnoEJ8zQT4Ky9OA2SfQi8gc9JRkJgcOixQxp4uzjdl5TdgpmduzUknfO6wJ8rpkm79Xc6QAuw4Yo/Ahud0wDsnwdqtLE4pxicGZhkAdHPxTgyVY8jHVq/u6hjrIgGAWC0rfCD27MPyl8oCrmUWtd/ubR2MQMzBoblbCgwIYh1wUVDrKhpGVRLIdAEBeP2DNxKuQeWcMpuz9RZqjPDP1lhTohDDYzObPzh6s27X64+/HXs2M5iOrWX/VNrp6oPgyObu32TSjDlSQN66dgn18izrDnNtXunAXvj6OBpb+exXfYAKhuVII3KFXlPcRpmT4gNhBx8jPWx2NM+lBGfAquXIIkRLSiYZE/LWGhg+Z4krBeNjhE8IUxyL6ZkoRgOPP5vzxvKEx9+mWixIMvpuV4MJDoCmpnwx9lTqvUczIyStV8z0Gfw0FLcpFpY50o0EoKGYMLt+1mnIEoMN7dnwrvrPd7hHpELHUM1TlUYQJILPDlS1qfXvn8VbwIeUMhQqgpMBF3nQZIFD+b+EBZji0nMQAbAww26KBki9ejHJvi/wdD6rpti6w8o7tYra2G3qP4IM7O9JliHeFNnvAW+WMK6gsHZErDdXOYEih13lHBwLedpnKKkRvtd1Pw7iQDNBlUHL7BNuA7FQDcXSm6D2bmaraSgUww4NlgqpAxA68env9WOXc3gAUWX0NQHOufKOKJzemusEwPoA4+qxFRQ+X2SNMyfAv1PSqfg8Lng1Hkzhm/7CIio+eFz2rTdFOg8bQJMkRYzwFz9lKXUIK9+jPwoDbsk5WJH0jhCM6g9oNy7tqsCsIvZkosSE2JlN92SdYfM7DhJvYiW4/5ljMPiTlEe6/XHd+/fX/3JPNmmKBmejR3DzKP7FgBfpXXIG+sanRAwP/uV8lQDNDmgNGRVyKUzMMzOIN8zaTTl6LgsvQUrFYPd0IkpD/W40Ij0EsIoEFIMfPIc/las1CJCEy0ksJfP1Uls1DG03djdCOMNM6LxqllmjPT8In24lC4LhL4jwYYgFk3VAV2ZzDxbm8B4DXRBCwGPAOSBCp8Lo0pt64PostNDh0WTEDcp2dG7tmR9//ant9d/QdFqGKZqktZPDGF16PX+8vraPOsIpiO4JfPvJolfYlkuk6RychwLSpeK/yUffWqhc2Nbw6vIAB+DYc6noYitg4QwZil7pdKXQTOfGdaSjw90D+TpeKxB0GuVhVoi96/04Ckut/WSCGjFpCoZCDqlratJn5kewf/uOmCbGZQA0RamO81Sq2KKkSckHTCaRDJH5QlFfwahrC+CWhd47nz3oCB1zaYg+AFWW9CoA/aJoKQOVBaHxjQuqBW4mtzYT3uOPGe2h1RtMJzwTXEah3VYHgEnp1fV5JvynIS0UBSJUpUXxgWEJfyL9Lzwky9c90hyMsahSJsKPSb0U8xKNmjZRcr2Cd/9qQ3FkQRuOs2aIKkIHensEbUOnh5x9FK0kSwrIcED4wJOPwFQZeqaJDT3yI9MUK8kEv/83rr6lCdxAIl7SHMKiXAagHZDooPFoCziS7378PbPb3+6/EHXfiz/UFIMsSYQEVgMYs2AjkykeQwGA2zSfZL5oLSRrJWB5wJlRYKx9AO2Ge0VrsC2Mfcssto5qXCh2ghJ8cDIeWq/AxHtbFBdRugi84yRpPqJ0sF/HdeuGm9qNRO11aruJAfEwtWaernh0GSsMlAXtUJhI1rSi8h/TDyPB67eqA73+dWrDlDBXAbQlhGmS60FJ9rBtucGYoxhObrbs9FJ0uAoCIgPa5HXtY0kY7woa4AOW9wBwIenCi0wqwOVCWH4L6W1EpCb1Yk4HB5pjqQGZlkpra0+qQvThgybOG/6FfvOOqrXxzAZcbXGUoWPHUAVsqpM2mPaRnI9YysqWVc5mjFRDdUTc3PSb8GBLly/OQ8aWxnpVY1NiVUOaTConXQVqmjeVX6au1kuUdx8NLIebogl+dxXaL88rDkijBeB6t2NqJrd8SgNP6DVw4k4AD9rAWabua1w88Vx/bt6FMfp3tGdTwu0t7KgEdbI1s4fs0iUnnotvZntmSdYS2VHnbHEqAphRo3JfIcd2XDDcEA+kOvCgwgjMDhCYeBDY36JxtqyrTIK68KsQPPAaJ8nzY1JHVZ1+4EyGNziA7nwjfFdwj+GNw/0cNeej6cNI3jYWMdL0bIXuHE49R04OY/Ff6fiN6/+exAnl56P5STWcuDN0AozjoILkhFLIR+DglK8SPAkidW85xN5e96R+jbzDJnOamZitif4V2RPN3qXfGt6KyDcSC9AcHrlSuKgYPS5hR5zKYFPFNC8+4KEMWR2nDCj9noU7/BrUuFKZRDduVrYguAAQyOKq8epyktqx3REu5vxOqx5aBlAmfWZFuOuFX92yCnHdiy3bE/BI8RZL0sum7P4uRQgq81CpRqlCckxlGrdwPBbGwbGLQ3bZRL8813DADWU4EsC5xakiHf4nbmwhyqB5UkjHFpcsirPFLV+rP1irltgOhp2MaWhmPJWvZa4cCcwxLD4Mij3WGeRJTp+a8dvQlGgCUzOYyDnz+9/hjjfp8mkoRn7Mt/rNE3aPVNcOoyzvNiAFQdi+kiSODwCzA+rH74qSDG0ps91Feg2S+I6pejSBq4kgIhL/jEtQvmX28W7VIwhLnC5xr6xCisioh039zd97KqtxRdpYCIwm9/bTzhDEDH3NGioJ3mJtntHcg/MHA9+LsD45Hv8+kTj+y3EqxkI2wUaqga3/TpiRetvgjuKaRIiPztU4PZsB+Y5aQX7YszHIu0xgCwHsQXLXvSMb2O8PTj0jBbpfc8II2hOIUs5CQEJOjbAiFj0KGSwL1jWR2iwD8mR1URiBeaN5j0QNM+woN7PxpMAkB+xY+OPWBcnQZF5pJcIUiSorfJysY8hJUn6xrDtZkfxjvO+yPZ5H0GgQELfPQAu4qAFNnyJn+I2LkkGVcPCgPDQaWT5/O+hGTIJMR52eqGEpoPPIkCA3d1TpcsyEiA3hijePXOv5nyDJzmTZhvtMtqAc6Gv8hIMeIi3LBGBXDJsxW0lsA6ZdtCJmDIieqQdq0oeagi8aqLkwXuE0+K9EcUjFioO/Gr7DpymCSkuzDmohnhl2ZOV+0V7buQOv3y4/NHawgYgn971bfZkXMNu7Lue2EZGBd12q845wKh4ZgQbVVgB23XmeIF5oxUVv00njttGXUUtKkY5Ep78wQiAMEY5Gs+0D7R9Myzn8hsExYHvrLnbvO89fV69d0W1YvjcHYueCHFZNGzWcbRjv0G/yo9SbK4GBx7CIqC1jWxXJK40zsv+UisfruVv8tuLTIhC/23l106b0kDduP+tUPbeHr8Aa8XfSmg4ZoPx32Kn3u/9RES0uiws728k37syfpow2ljFKLb01O4lwg5v8bkv8FbpyUZnGKM+0BCyLfRnG+mqj4DqK/hN7WKgF77WF7CpNyQOhuYdcv+S5CAMhcbCV1Ztr9xaN0DujmyVmXfSm9oFerOkOexF05TGzQlh7eePzJ43eIsr/FztyKwqW6x/5Y7Y4+29sVDDavy5dzmeZXvVonmcDoyy7ekCzzGOsBYfanW0YzLVUtONqaO9M1u+aFNr8RmfrHn1b6fH1m2OGbrT2Ixi6+YF9dgjalVvnFP58oaXafqPnwOr3leYBWFOKebxpriOic/9oWet0qu6OrElfdCs4vJGcLxeVE3hk8vifo99UO/5CF7yBAUYOMy8PC/MAs8bmlMnJIQTk3PA8o7Hqk9pxLvEL0TzaAFhblzQ0MzT+hA8ZcUDywmP478Wh2ze+gYM9/l+LCVyTEqNKAKVL78MkzIeI4sEMoNF8vC491XhrG82bybBZxABZWwsG0++FpmS5XFDeb6aS90Ix8q8diMDDBjeSpz8L8TKBvXWQfw0kRkGdnWoT5wjnuSIJznyooaOj9juUrXCYOEAizolBIL4mEO39ejaV6s1RnYVVx3SkshOo2MdGZQeSMJIce3sCTzimV/W5oeuhK+iFUuuKjR1eLLHkAODPhgW+rdo3Gs9xJF9e7yPVDTZyIc0uo1en8Pvrcsi2GKjBz7BqbeD8+cZsjsHW8O5n4RT9cUTIV7h1M9wkCcghOrGi4EsCCy19KvOPu6DRTGQneMNQBGjQrBz3uz7y9RVLeCL8279kD5bMktpGI+UhCnuk5oJ7kpks4ayGGTrfIPzxRwwZ/JIW6z3LX33sh3YrzUy6CMyF8d76bSc7B7CuBiIH4xbBHxWAAR72YNpIMypIgQXhEuaJZR+F3LR7qlqHJ1sVeg70OaAVMfm57oKjOqX3Dt60fDwnXbKBGrso37m0socA6lO4qJTyQ02NpdqXEcCAn4DrMfF5+YsDdxoETkx7WgkVb+GfGHDpDmhr3PyBcZXv3zpJFd3Y+lOEiS2sdeuuZM8ywe6xXXE462q4xg9p0E+fwhptLEKejfiosVi8Sej8/A1FkR2MW9tfm14KW4wc1UPm9SyxCjZs+1RqdunKkwEW+J52D2JHSh4AeZ5aH89TzfaiSDyNoVdnbVNGr7y5IcpKlubI09jRgLMzIWrJzdykL+2EGMQEGcMn4gdxvKrhFGTPJVo4RLj6XRsLMJnePxo0E7XVhtZ23/9LzqGf//zH4G1jf/9z/9Orf/7n/h3FYn4IESm4DJW8Nq91Hhz5unG/9bzmTT05q58heMZL9tyvUztdYHmHDqTN+9+uPyjdikm67Tf3VjT5Xq2Xq3Xa3fiztz5fLmSYMqlb5SaYVIOdkVt6AVOjH6iwZ7HdQKZajJR+CqdA/bPpm7oO65j+wFZL9bharoK6cIPXGc5dWarFXWchesGq2hhr5fTKJjZNKIEJjnUd135aOpZEq8aDb6JfIVkciC7pIfsmT2j7mrhrpYRXU2d5RJYSUN/sabr+XRF7CBcheHMdufrWeTb0doJyYLMZi4htmOvlrROdqO2cnu2WK1sZwnIl5TOwtXasZezKJov6GLhLsO5G9nhch46q1lEfJesSDhzV05kL9dA0CJw1vrQG0+AEDeQ65LZ2ier9WzhzJcze7qeEWe6JnN/RvzZ3HbXwNvFgjrRAkiY+cEUOO3PoqW/dv2pxt0undyerWGPUzu0lwSoXIbOLHTcYO467nIxd9dTCjjp3A3o3I7C+doNbOq49tL316ul48Bha+RtK86VcEkDOwTIZWjPA39q+zPgEqXraLmioTN1/PVs5UdBsHSmdjSn8BdduFPgWjj1SVDxpe8FBS4SgjL4kT1d+CvfXQEzFoAmgv/N7WnkBMCsdbAMZ6sl7Gg5n86dpWvPF8v5Yjb3bTLtWKT1ggKXAfrpIrBdB/bhr2waTIPFdBY4wcJxfeC8u4QfZOnPQfrnMxqQKbXtaBpNw2U0XUR+pdhGbWZj3Shx1YJ9/eO7/7oav7HH3//8ww+GPLcBPlxd//zjVT/I9fwEDgA4heOP8/nqBBYOovAgzJ2SuM53kPpRauvN4wZfgDXn1t83bvBxo5qtyimqaUuBefiYzxOtG+JtHk5cTNXEL3sEubFK5V25o6Kk9Or1HISJdBlH0F4aTkQ8nKvDGLWgIOalQZkB1cFOPq6sgwtXiLLlqRRIbeA2hUDo7Pn/ASkKAKw='
RECOVERY_PAYLOAD_SHA256 = 'd1c0bec0e18f80784cce824ac20f13395155c7840c4143f462efd47cada84027'
attempt_started_at = time.time()
watchdog = None
worker = None

def set_deadline(deadline):
    global watchdog
    if watchdog is not None:
        watchdog.cancel()
    watchdog = threading.Timer(max(1.0, deadline - time.time()), runtime.unassign)
    watchdog.daemon = True
    watchdog.start()

set_deadline(attempt_started_at + 12 * 3600)
try:
    drive.mount('/content/drive')
    storage = Path('/content/drive/MyDrive/E009')
    archive = storage / 'e009-colab-execution-bundle.zip'
    journal = storage / 'E009-launch.json'
    first = attempt_started_at
    previous_setup = 0.0
    if RESUME:
        if not OLD_RUNTIME_STOPPED or not journal.is_file():
            raise RuntimeError('Resume requires the original journal and a stopped old runtime.')
        state = json.loads(journal.read_text(encoding='utf-8'))
        if state['bundle_sha256'] != EXPECTED_SHA256 or state.get('setup_in_progress'):
            raise RuntimeError('Bundle mismatch or interrupted setup; preserve existing files for review.')
        first = float(state['started_at'])
        previous_setup = float(state['setup_elapsed_seconds'])
        if not math.isfinite(first) or not 0 < first <= attempt_started_at:
            raise RuntimeError('Invalid original campaign timestamp.')
        if not math.isfinite(previous_setup) or previous_setup < 0:
            raise RuntimeError('Invalid cumulative setup time.')
    elif any(p.exists() for p in (journal, storage / 'E009-V15-COLAB-20261106',
                                      Path('/content/e009-workspace'), Path('/content/e009-venv'))):
        raise RuntimeError('E009 already exists. Verify the old process stopped, then use recovery mode.')
    deadline = first + 12 * 3600
    set_deadline(deadline)
    setup_deadline = min(deadline, attempt_started_at + 40 * 60 - previous_setup)
    if not RESUME:
        setup_deadline = min(setup_deadline, first + 40 * 60)

    def setup_remaining():
        remaining = setup_deadline - time.time()
        if remaining <= 0:
            raise TimeoutError('Original campaign/setup budget exhausted; timestamps must not reset.')
        return remaining

    setup_remaining()
    if shutil.disk_usage('/content').free < 50_000_000_000:
        raise RuntimeError('At least 50 GB of local disk must be free.')
    with archive.open('rb') as stream:
        digest = hashlib.sha256()
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
        actual_sha256 = digest.hexdigest()
    if actual_sha256 != EXPECTED_SHA256:
        raise RuntimeError('Bundle SHA mismatch; use the handed-off execution bundle.')

    # Install only bootstrap tooling beside the kernel, then real managed CPython.
    tools_dir = Path('/content/e009-python-tools')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                    '--no-cache-dir', '--no-deps', '--target', str(tools_dir),
                    'uv==' + UV_VERSION], check=True, timeout=setup_remaining())
    tool_env = dict(os.environ, PYTHONPATH=str(tools_dir),
                    UV_PYTHON_INSTALL_DIR='/content/e009-managed-python')
    uv_command = [sys.executable, '-m', 'uv', '--no-config']
    subprocess.run(uv_command + ['python', 'install', PYTHON_VERSION, '--no-bin'],
                   env=tool_env, check=True, timeout=setup_remaining())
    python = subprocess.check_output(
        uv_command + ['python', 'find', '--managed-python', '--no-python-downloads', PYTHON_VERSION],
        env=tool_env, text=True, timeout=setup_remaining()).strip()
    worker_code = "\nimport base64, hashlib, json, subprocess, sys, types, zlib\nfrom pathlib import Path\narchive, sha, storage, started, resume, stopped, packed, payload_sha = json.loads(sys.argv[1])\nif sys.version_info[:3] != (3, 12, 12):\n    raise RuntimeError('A real CPython 3.12.12 interpreter is required.')\nif not resume or not stopped:\n    raise RuntimeError('Recovery requires RESUME and confirmation the old runtime stopped.')\npayload = zlib.decompress(base64.b64decode(packed, validate=True))\nif hashlib.sha256(payload).hexdigest() != payload_sha:\n    raise RuntimeError('Recovery payload SHA mismatch.')\nfiles = json.loads(payload)\nif set(files) != {'extension.py', 'authorization.json'}:\n    raise RuntimeError('Unexpected recovery payload members.')\nauth_bytes = files['authorization.json'].encode('utf-8')\nauth = json.loads(auth_bytes)\nextension_bytes = files['extension.py'].encode('utf-8')\nif hashlib.sha256(extension_bytes).hexdigest() != auth['extension_sha256']:\n    raise RuntimeError('Recovery extension SHA mismatch.')\nauth_sha = hashlib.sha256(auth_bytes).hexdigest()\nextension_path = Path('/content/e009-recovery-20261107.py')\nauth_path = Path('/content/e009-recovery-authorization-20261107.json')\nbootstrap = Path(storage) / 'E009-bootstrap'\nbootstrap.mkdir(exist_ok=True)\nfor target, content in ((extension_path, extension_bytes), (auth_path, auth_bytes),\n                        (bootstrap / extension_path.name, extension_bytes),\n                        (bootstrap / auth_path.name, auth_bytes)):\n    if target.exists() and target.read_bytes() != content:\n        raise RuntimeError('Recovery input differs from existing pinned bytes: ' + str(target))\n    target.write_bytes(content)\nprint('Launcher/training base:', sys.version, sys.executable, flush=True)\nprint('Recovery extension:', auth['extension_sha256'], 'authorization:', auth_sha, flush=True)\nsys.path.insert(0, archive + '/src')\nfrom ai_exam_monitoring.training.colab_launcher import launch\noriginal_run = subprocess.run\n\ndef route_campaign(command, **kwargs):\n    if command[1:3] == ['-m', 'ai_exam_monitoring.training.representation_campaign']:\n        if '--resume' not in command or '--prior-process-stopped' not in command:\n            raise RuntimeError('The recovery extension cannot start a fresh campaign.')\n        command = [command[0], str(extension_path), *command[3:],\n                   '--recovery-authorization', str(auth_path),\n                   '--recovery-authorization-sha256', auth_sha]\n    return original_run(command, **kwargs)\n\n# Reuse the original launcher's verified setup/closure/clock/export. Only its final\n# campaign entry point is rebound; no package source or global function is edited.\nclass ProcessAPI:\n    run = staticmethod(route_campaign)\n\n    def __getattr__(self, name):\n        return getattr(subprocess, name)\n\nprocess_api = ProcessAPI()\nrecovery_launch = types.FunctionType(launch.__code__,\n    dict(launch.__globals__, subprocess=process_api), launch.__name__,\n    launch.__defaults__, launch.__closure__)\nrecovery_launch.__kwdefaults__ = launch.__kwdefaults__\nrecovery_launch(Path(archive), sha, Path(storage), attempt_started_at=started,\n                resume=resume, prior_process_stopped=stopped)\n"
    arguments = json.dumps([str(archive), EXPECTED_SHA256, str(storage),
                            attempt_started_at, RESUME, OLD_RUNTIME_STOPPED,
                            RECOVERY_PAYLOAD, RECOVERY_PAYLOAD_SHA256])
    setup_remaining()
    print('Colab kernel:', sys.version, flush=True)
    worker = subprocess.Popen([python, '-I', '-u', '-c', worker_code, arguments],
                              start_new_session=True, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1)
    def forward_worker_output():
        if worker.stdout is not None:
            for line in worker.stdout:
                print(line, end='', flush=True)

    output_reader = threading.Thread(target=forward_worker_output, daemon=True)
    output_reader.start()
    returncode = worker.wait(timeout=max(1.0, deadline - time.time()))
    output_reader.join(timeout=5)
    if returncode:
        raise RuntimeError(f'E009 launcher stopped (exit {returncode}); inspect its output/metadata.')
finally:
    try:
        if worker is not None and worker.poll() is None:
            try:
                os.killpg(worker.pid, signal.SIGKILL)
                worker.wait(timeout=10)
            except (OSError, subprocess.TimeoutExpired):
                pass
    finally:
        try:
            runtime.unassign()
        except Exception:
            print('Select Disconnect and delete runtime if Colab has not disconnected.')
        finally:
            if watchdog is not None:
                watchdog.cancel()
